# DFT & DCT on real vs. "fake" (upsampled) images

This notebook is a hands-on companion to the discussion of Durall et al. (2020) and
Frank et al. (2020) on why GAN/diffusion-generated images carry detectable
frequency-domain artifacts.

**What we do:**
1. Take one real photo.
2. Simulate a "fake" image the way a generator would create one: shrink it down to a
   small "latent" resolution, then blow it back up using two common upsampling
   methods (nearest-neighbor, which behaves like the zero-insertion/transposed-conv
   case, and bilinear interpolation).
3. Compute the DFT and DCT of all three images and compare their frequency content.

**Important caveat:** this is a *simplified simulation* for intuition, not a real
trained GAN. A real generator's artifacts come from learned convolution kernels
across many layers. Here we isolate just the upsampling step so the effect is easy
to see and reason about.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, transform
from scipy.fft import fft2, fftshift, dctn

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 6)

## 1. Load a real image

We use a built-in sample photo from `scikit-image` (no download needed) and convert
it to grayscale, since frequency-domain analysis is easiest to reason about on a
single channel.

In [ ]:
real = color.rgb2gray(data.astronaut())
real = transform.resize(real, (256, 256), anti_aliasing=True)

plt.imshow(real, cmap="gray")
plt.title("Real image (256x256)")
plt.axis("off")
plt.show()

## 2. Build the "latent" and two fake upsampled versions

We shrink the real image down to 64x64 to stand in for a generator's small latent
feature map, then upsample it back to 256x256 two different ways:

- **Nearest-neighbor upsampling**: each latent pixel is duplicated into a block of
  identical pixels. This creates sharp block edges — conceptually the same family
  of artifact as transposed convolution's zero-insertion: information is not
  smoothly reconstructed, so you get abrupt jumps that inject excess high-frequency
  energy.
- **Bilinear interpolation**: each output pixel is a weighted average of its
  neighbors. This is a low-pass filter, so it smooths out fine detail — the
  "up + conv" / interpolation case that suppresses high frequencies.

In [ ]:
latent = transform.resize(real, (64, 64), anti_aliasing=True)

fake_nn = transform.resize(latent, (256, 256), order=0, anti_aliasing=False)       # nearest-neighbor
fake_bilinear = transform.resize(latent, (256, 256), order=1, anti_aliasing=False)  # bilinear

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [real, fake_nn, fake_bilinear],
    ["Real", "Fake: nearest-neighbor\n(transposed-conv-like)", "Fake: bilinear\n(interpolation-like)"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. DFT: 2D frequency spectrum

We take the 2D Discrete Fourier Transform of each image, shift the zero frequency
to the center, and look at the log-magnitude. Low frequencies live near the center;
high frequencies live near the edges/corners.

Real images should show a spectrum that's bright in the center and gradually fades
outward. Watch for:
- **Nearest-neighbor fake**: extra brightness / periodic structure further out
  toward the edges (excess high-frequency energy from the blocky jumps).
- **Bilinear fake**: spectrum fades out faster than the real one (missing
  high-frequency detail).

In [ ]:
def dft_log_magnitude(img):
    F = fftshift(fft2(img))
    return np.log(np.abs(F) + 1e-8)

spectra = {name: dft_log_magnitude(img) for name, img in
           [("Real", real), ("Fake (nearest-neighbor)", fake_nn), ("Fake (bilinear)", fake_bilinear)]}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
vmin = min(s.min() for s in spectra.values())
vmax = max(s.max() for s in spectra.values())
for ax, (name, spec) in zip(axes, spectra.items()):
    im = ax.imshow(spec, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"DFT log-magnitude\n{name}", fontsize=10)
    ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
plt.show()

## 4. Radial (azimuthal) average spectrum

A single 2D spectrum image is hard to compare precisely by eye. Papers like Durall
et al. and Frank et al. instead average the spectrum's energy over rings of equal
distance from the center ("radial" or "azimuthal" average), which collapses the 2D
spectrum into a single 1D curve: energy vs. frequency (distance from center = 0
frequency).

A natural image's curve falls off smoothly. Watch where the fake curves diverge
from the real curve, especially in the mid-to-high frequency range (right side of
the plot).

In [ ]:
def radial_profile(mag2d):
    h, w = mag2d.shape
    cy, cx = h // 2, w // 2
    y, x = np.indices((h, w))
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2).astype(int)
    total = np.bincount(r.ravel(), mag2d.ravel())
    count = np.bincount(r.ravel())
    return total / np.maximum(count, 1)

profiles = {name: radial_profile(spec) for name, spec in spectra.items()}

plt.figure(figsize=(7, 5))
for name, rp in profiles.items():
    plt.plot(rp, label=name)
plt.xlabel("Frequency (distance from center, 0 = DC / low-freq)")
plt.ylabel("Log-magnitude (avg. over ring)")
plt.title("Radial-average frequency spectrum")
plt.legend()
plt.tight_layout()
plt.show()

## 5. DCT: the other frequency lens

The Discrete Cosine Transform is the transform JPEG compression is built on. Unlike
the DFT, it produces purely real-valued coefficients. We apply it here the same way
Frank et al. (2020) do — as a whole-image 2D transform — to see the same real-vs-fake
divergence show up in a second, independent frequency representation.

In the DCT image below, the top-left corner is the lowest frequency (roughly average
brightness), and moving toward the bottom-right corresponds to progressively higher
frequencies (fine detail / noise).

In [ ]:
def dct_log_magnitude(img):
    d = dctn(img, norm="ortho")
    return np.log(np.abs(d) + 1e-8)

dct_maps = {name: dct_log_magnitude(img) for name, img in
            [("Real", real), ("Fake (nearest-neighbor)", fake_nn), ("Fake (bilinear)", fake_bilinear)]}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
vmin = min(d.min() for d in dct_maps.values())
vmax = max(d.max() for d in dct_maps.values())
for ax, (name, d) in zip(axes, dct_maps.items()):
    im = ax.imshow(d, cmap="magma", vmin=vmin, vmax=vmax)
    ax.set_title(f"DCT log-magnitude\n{name}", fontsize=10)
    ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
plt.show()

## 6. Wrap-up

- **DFT and DCT** are two different mathematical lenses for the same question: "how
  much energy does this image have at each frequency?"
- **Real images** have a smooth, predictable falloff from low to high frequency.
- **Nearest-neighbor upsampling** (standing in for zero-insertion / transposed
  convolution) introduces sharp block edges, which shows up as *extra* high-frequency
  energy relative to a smoothly-downsampled real image.
- **Bilinear interpolation** smooths the image out, which shows up as *missing*
  high-frequency energy — the spectrum falls off faster than it should.
- This divergence — in either direction — is exactly the signal that lets a simple
  classifier trained on frequency coefficients (like in Frank et al., 2020) separate
  real images from GAN/diffusion outputs with very high accuracy.

Try swapping in your own image (replace the `real` array above) or changing the
latent size (e.g. 32x32 vs 64x64) to see how the effect gets more or less severe.